# 3.1 — 表形式データ・CSV・pandas

2.3で扱った辞書のリストを表へ移し、CSVを読み込み、列の意味と型を確認してから計算へ進みます。

## 導入

このNotebookでは、Moodle本文の概念を実際のデータとコードで確かめます。

## このレッスンの到達目標

- 表形式データの一行・一列・セルが何を表すか説明できる。
- CSVの読込条件と実際に読み込んだファイルを確認できる。
- DataFrameの形、列名、型、欠損、カテゴリ値を調べられる。
- 計算列を追加し、indexを混入させずCSVへ保存できる。

> **学習経路:** 必須：3.1.1〜3.1.4　／　補足：3.1.5　／　統合練習：3.1.6


## 3.1.1 行・列・セルとスキーマを理解する

この教材では、一行を一つのセンター・月の観測、一列を同じ意味を持つ変数、一つのセルを一観測の一変数の値として扱います。最初の行の列名と、各列に期待する型・単位・欠損規則を合わせてスキーマと考えます。

In [ ]:
import pandas as pd

records = [
    {"month": "2026-01", "centre_id": "C001", "registered": 32, "completed": 24},
    {"month": "2026-01", "centre_id": "C002", "registered": 27, "completed": 18},
]

records_df = pd.DataFrame(records)
records_df


## 3.1.2 CSVと読込条件を理解する

CSVでは通常、一行目がヘッダー、後続行がレコード、カンマがフィールドの区切りです。値にカンマや改行を含む場合は引用符が必要です。文字コード、区切り文字、欠損を表す空欄、先頭0を保持したい識別子を意識して読み込みます。

### 相対パスと実際に読み込むファイル

`data/file.csv`はNotebookファイルの場所ではなく、カーネルの現在の作業フォルダから解釈されます。Python Labを別の入口から開いても教材CSVを読めるよう、次の補助関数は現在位置、その親、サーバー上の学習領域、配布元を順に調べます。表示される`Working directory`と`Loading`を必ず確認してください。

In [ ]:
from pathlib import Path
import pandas as pd


def find_course_data(filename):
    """Find course data without depending on the Notebook start directory."""
    roots = [
        Path.cwd(),
        *Path.cwd().parents,
        Path.home() / "work",
        Path("/opt/python-lab/course-materials"),
    ]
    checked = []
    for root in roots:
        for candidate in (root / "data" / filename, root / filename):
            candidate = candidate.expanduser()
            if candidate in checked:
                continue
            checked.append(candidate)
            if candidate.is_file():
                return candidate
    locations = "\n".join(f"- {path}" for path in checked)
    raise FileNotFoundError(
        f"Course data file {filename!r} was not found. Checked:\n{locations}"
    )


data_file = find_course_data("learning-centres-practice.csv")
print("Working directory:", Path.cwd())
print("Loading:", data_file.resolve())


### read_csvで読み込み方を明示する

`pandas`は慣例的に`pd`という名前で読み込みます。`read_csv()`はCSVから`DataFrame`を作ります。ここではUTF-8を明記し、コードや年月を計算対象ではない文字列として保持します。他のデータでは区切り文字や文字コードが異なることがあります。

In [ ]:
df = pd.read_csv(
    data_file,
    encoding="utf-8",
    dtype={"centre_id": "string", "month": "string"},
)
print(df.head(3))


## 3.1.3 DataFrameを読み込み、内容を確認する

`head()`で値の並び、`shape`で行数と列数、`columns`で列名、`dtypes`と`info()`で推定型、`isna().sum()`で欠損数を確認します。想定と違う場合は、計算を始めず読み込み方か入力データを調べます。

### 全件表示とカテゴリ値の件数を確認する

`head()`は先頭確認用です。人が確認できる件数なら`to_string(index=False)`で全件を省略せず表示します。補正前のカテゴリ値は`value_counts(dropna=False, sort=False)`で数え、必要なら名前付き2列の表へ変換します。

In [ ]:
print(df.to_string(index=False, line_width=200))

district_counts = (
    df["district"]
    .value_counts(dropna=False, sort=False)
    .rename_axis("district")
    .reset_index(name="records")
)
print(district_counts.to_string(index=False, formatters={"district": repr}))


In [ ]:
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Dtypes:")
print(df.dtypes)
print("Missing values:")
print(df.isna().sum())
print("Info:")
df.info()


### 一列はSeries、複数列はDataFrame

`df["registered"]`は一次元の`Series`を返します。`df[["registered"]]`は一列を持つ二次元の`DataFrame`です。列名は文字列として正確に指定し、複数列では外側と内側の二組の角括弧を使います。

In [ ]:
registered_series = df["registered"]
registered_table = df[["registered"]]
print(type(registered_series).__name__, registered_series.shape)
print(type(registered_table).__name__, registered_table.shape)


## 3.1.4 計算列を作り、表を保存する

Seriesどうしの演算は行の対応を保って全行へ適用されます。`assign()`を使うと元の`df`を直接変更せず、計算列を追加した新しいDataFrameを作れます。分母0や不正値の扱いは3.3で詳しく検討します。

In [ ]:
report = df.assign(
    completion_rate=df["completed"] / df["registered"] * 100
)
print(report[["month", "centre_name", "registered", "completed", "completion_rate"]].head())


### DataFrameのindexと業務上の識別子を区別する

左端のindexはpandasが各行へ付けるラベルで、`centre_id`の代わりではありません。CSVへ保存するときに`index=False`を指定すれば、不要なindex列を書き出しません。保存後はパスと先頭行を確認します。

In [ ]:
preview_file = Path.cwd() / "monthly-centres-preview.csv"
report.head(5).to_csv(preview_file, index=False, encoding="utf-8")
print("Saved:", preview_file.resolve())
print(preview_file.read_text(encoding="utf-8").splitlines()[0])


## 3.1.5 読込問題を原因別に切り分ける

`FileNotFoundError`なら表示された現在位置と探索先、列が一列だけなら区切り文字、文字化けや`UnicodeDecodeError`なら文字コード、数値列が文字列なら単位記号・空白・不正値を確認します。読み込めたことと、正しく読めたことは同じではありません。

## 3.1.6 統合練習：表を作り、保存し、再確認する

2.3のセンターレコードからDataFrameを作り、CSVへ保存して再度読み込んでください。読み込み前後で`shape`、列名、識別子、人数合計が一致することを確認し、`attendance_rate`と`completion_rate`を追加します。次の3.2では、この表から条件に合う行と必要な列を選びます。

In [ ]:
# ここに応用練習の解答を書きます。


## まとめ

- 一行の観測単位と列の意味を決めてから表を扱いました。
- CSVの場所と読込条件を確認し、DataFrame全体を観察しました。
- 計算結果を別の表として作り、保存後の形と列を確かめました。

## 次のレッスンへ

確認できる表ができました。3.2では、分析の問いを列と行条件へ分け、必要なレコードだけを再現可能な方法で選びます。

**学習時間の目安:** 約4時間
